In [ ]:
# RECOMMENDATION SYSTEM USING KNNs

# Import libraries
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Load CSVs
df = pd.read_csv("movies.csv")

# Remove duplicates
df = df.drop_duplicates(subset = "name", keep="first").reset_index(drop=True)

# Process Genres
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(df["genres"])

# Process PG rating
pg_enc = OneHotEncoder()
pg_matrix = pg_enc.fit_transform(df[["movie_rated"]])

# Combine all features
numerical_rating = df[["rating"]].values
combined_features = hstack([genre_matrix, pg_matrix, numerical_rating])

# Build a similarilty matrix
similarity = cosine_similarity(combined_features)

# Define recommendation function
def recommend_movie(movie_title, top_n = 5):
    movie_title = movie_title.strip().lower()
    df["clean_title"] = df["name"].str.strip().str.lower()

    matched = df[df["clean_title"] == movie_title]
    if matched.empty:
        return f"Movie {movie_title} not found."
        
    index = matched.index[0]
    sim_scores = list(enumerate(similarity[index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse = True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    return df.iloc[movie_indices][["name", "genres", "rating", "movie_rated"]]

# Prompt for user input
i = 1
while i > 0:
    user_input = input("Enter a movie title (0 to quit): ")
    if user_input == "0":
        i = -1
    else:
        recommendations = recommend_movie(user_input, top_n = 5)
        print(recommendations)

Enter a movie title (0 to quit):  8 Mile


                     name          genres  rating movie_rated
781       Velvet Goldmine  Drama; Music;      7.0           R
747      Black Snake Moan  Drama; Music;      6.9           R
741  Saturday Night Fever  Drama; Music;      6.8           R
714          Billy Elliot  Drama; Music;      7.7           R
770             Rock Star  Drama; Music;      6.3           R


Enter a movie title (0 to quit):  Blade Runner 2049


                    name                      genres  rating movie_rated
467               Oldboy    Action; Drama; Mystery;      8.4           R
470             Watchmen    Action; Drama; Mystery;      7.6           R
809          Cloud Atlas    Action; Drama; Mystery;      7.4           R
808  The Lives of Others  Drama; Mystery; Thriller;      8.4           R
820            Chinatown  Drama; Mystery; Thriller;      8.2           R


Enter a movie title (0 to quit):  The Dark Knight


                    name                      genres  rating movie_rated
398            Fast Five  Action; Adventure; Crime;      7.3       PG-13
417                  RED     Action; Comedy; Crime;      7.0       PG-13
380      Minority Report    Action; Crime; Mystery;      7.6       PG-13
189  Catch Me If You Can   Biography; Crime; Drama;      8.1       PG-13
402      The Italian Job   Action; Crime; Thriller;      7.0       PG-13


Enter a movie title (0 to quit):  John Wick


                     name                     genres  rating movie_rated
397  John Wick: Chapter 2  Action; Crime; Thriller;      7.5           R
404         The Equalizer  Action; Crime; Thriller;      7.2           R
423               Con Air  Action; Crime; Thriller;      6.9           R
370     Kill Bill: Vol. 2  Action; Crime; Thriller;      8.0           R
360     Kill Bill: Vol. 1  Action; Crime; Thriller;      8.1           R


Enter a movie title (0 to quit):  0]


Movie 0] not found.
